In [1]:
import numpy as np
from scipy.linalg import svd

In [ ]:
def spd_inv_split(W, epsilon=1e-10, method='QR', canonical_signs=False):
    """
    Compute :math:`W^{-1} = L L^T` of the symmetric positive-definite matrix :math:`W`.

    by first reducing W to a low-rank approximation that is truly spd.

    Parameters
    ----------
    W : ndarray((m,m), dtype=float)
        Symmetric positive-definite (spd) matrix.
    epsilon : float
        Truncation parameter. Eigenvalues with norms smaller than this cutoff will
        be removed.
    method : str
        Method to perform the decomposition of :math:`W` before inverting. Options are:

        * 'QR': QR-based robust eigenvalue decomposition of W
        * 'schur': Schur decomposition of W

    canonical_signs : boolean, default = False
        Fix signs in L, s. t. the largest element of in every row of L is positive.


    Returns
    -------
    L : ndarray((n, r))
        Matrix :math:`L` from the decomposition :math:`W^{-1} = L L^T`.

    """
    if _np.shape(W)[0] == 1:
        if W[0, 0] < epsilon:
            raise ZeroRankError(
                'All eigenvalues are smaller than %g, rank reduction would discard all dimensions.' % epsilon)
        L = 1. / _np.sqrt(W[0, 0])
    else:
        if epsilon == 0.:
            sm, Vm = _np.linalg.eigh(.5 * (W + W.T))
            sm, Vm = sort_eigs(sm, Vm)
            # threshold eigenvalues to be >= 0 and sqrt of the eigenvalues to be >= 1e-16 so that no
            # division by zero can occur
            L = _np.dot(Vm, _np.diag(1.0 / _np.maximum(_np.sqrt(_np.maximum(sm, 1e-12)), 1e-12)))
        else:
            sm, Vm = spd_eig(W, epsilon=epsilon, method=method, canonical_signs=canonical_signs)
            L = _np.dot(Vm, _np.diag(1.0 / _np.sqrt(sm)))

    # return split
    return L

In [ ]:
epsilon = 1e-6

L0 = spd_inv_split(covariances.cov_00, epsilon=epsilon)
rank0 = L0.shape[1] if L0.ndim == 2 else 1
Lt = spd_inv_split(covariances.cov_tt, epsilon=epsilon)
rankt = Lt.shape[1] if Lt.ndim == 2 else 1

W = np.dot(L0.T, covariances.cov_0t).dot(Lt)
from scipy.linalg import svd
A, s, BT = svd(W, compute_uv=True, lapack_driver='gesvd')

singular_values = s

m = CovarianceKoopmanModel.effective_output_dimension(rank0, rankt, dim, var_cutoff, singular_values)

U = np.dot(L0, A[:, :m])
V = np.dot(Lt, BT[:m, :].T)